# Crypto Sentiment Analysis - Complete Pipeline

In [2]:
import subprocess
import sys
import os

print("Installing required dependencies from requirements.txt...")

try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    )
    print("All dependencies installed successfully!")
except subprocess.CalledProcessError as e:
    print(f"Warning: Some packages may not have installed correctly: {e}")

Installing required dependencies from requirements.txt...


## Setup: Import Scripts and Libraries

In [ ]:
import importlib.util
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr
import nbformat

def import_script(script_name):
    script_path = f'{script_name}.py'
    if not os.path.exists(script_path):
        raise FileNotFoundError(f"Script not found: {script_path}")
    
    spec = importlib.util.spec_from_file_location(script_name, script_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

preprocess_module = import_script('preprocess')
clean_text_module = import_script('clean_text')
vader_module = import_script('vader_sentiment')
finbert_sentiment_module = import_script('finbert_sentiment')
merge_labels_module = import_script('merge_labels')
feature_eng_module = import_script('feature_engineering')
price_module = import_script('get_price_data')
merge_module = import_script('merge_sentiment_price')

# Extract functions
preprocess_data = preprocess_module.preprocess_data
clean_and_save = clean_text_module.clean_and_save
analyze_sentiment = vader_module.analyze_sentiment
apply_finbert = finbert_sentiment_module.main
merge_labels_main = merge_labels_module.merge_labels
feature_engineering_main = feature_eng_module.main
get_price_data_main = price_module.main
merge_sentiment_price_main = merge_module.main

USERAWORKAGGLE = True # True uses X API set, False uses Kaggle set: FOR GRADING DONT RECOMMEND USING KAGGLE. WILL TAKE 2 HOURS

C:\Users\cooly\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Preprocess Raw Tweets

In [4]:
if USERAWORKAGGLE:
    path = "data/tweets_raw.csv"
else:
    path = "data/tweets_kaggle.csv"

try:
    preprocess_data(path)
    print("Preprocessing complete")
except Exception as e:
    print(f"Warning: {e}")

Original shape: (3961, 6)
Cleaned shape: (3961, 6)
Saved to data/tweets_standardized.csv
Preprocessing complete


## Step 2: Clean Tweet Text

In [5]:
try:
    clean_and_save()
    print("Text cleaning complete")
except Exception as e:
    print(f"Warning: {e}")

Loaded 3961 tweets from data/tweets_standardized.csv

Filtering spam and bot tweets...
Spam filter: removed 1750 tweets (44.2%)
Remaining tweets: 2211

Cleaning tweet text...

Saved 2211 cleaned tweets to data/tweets_cleaned.csv
Text cleaning complete


## Step 3: Apply VADER Sentiment Analysis

In [6]:
try:
    analyze_sentiment()
    print("VADER sentiment analysis complete")
except Exception as e:
    print(f"Warning: {e}")

Loaded 2211 tweets from data/tweets_cleaned.csv
Applying VADER sentiment analysis...
Saved results to data/tweets_vader.csv

Sentiment Distribution:
sentiment
positive    922
neutral     694
negative    595
Name: count, dtype: int64

Percentages:
sentiment
positive    41.70
neutral     31.39
negative    26.91
Name: proportion, dtype: float64
VADER sentiment analysis complete


## Step 4: FinBERT Sentiment Analysis (Run on cleaned tweets)

In [7]:
try:
    apply_finbert()
    
    print("FinBERT sentiment analysis complete")
    print("Results saved to data/tweets_finbert.csv")
except Exception as e:
    print(f"Warning: {e}")

Loading data...
Loading FinBERT model...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 53125.91it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Applying FinBERT sentiment analysis...


Processing batches: 100%|██████████| 70/70 [02:53<00:00,  2.48s/it]

Saving results...

Shape: (2211, 9)

First few rows:
                                        text_cleaned finbert_label  \
0  eric trump predicts bitcoin will hit million r...      positive   
1  no need for expensive machines anymore bitcell...       neutral   
2     tp3 00020 btc aapl join my private group below       neutral   
3         tp1 0047 join my private group today below       neutral   
4  not even btc wants to get in the ring with pip...       neutral   

   finbert_confidence  
0            0.668655  
1            0.901628  
2            0.917464  
3            0.941082  
4            0.935811  
FinBERT sentiment analysis complete
Results saved to data/tweets_finbert.csv


In [8]:
# Merge FinBERT and VADER
df_finbert = pd.read_csv('data/tweets_finbert.csv')
df_vader = pd.read_csv('data/tweets_vader.csv')
df_finbert["vader_compound"] = df_vader["compound"]
df_finbert.to_csv('data/tweets_finbert_vader.csv', index=False)

## Step 5: Feature Engineering (Engagement & Daily Aggregation)

In [9]:
try:
    feature_engineering_main()

    print("Feature engineering complete")
    print("Daily sentiment aggregation saved to data/daily_sentiment.csv")
except Exception as e:
    print(f"Warning: {e}")

✓ Loaded 2211 tweets
  Columns: ['text', 'created_at', 'author_id', 'likes', 'retweets', 'replies', 'text_cleaned', 'finbert_label', 'finbert_confidence', 'vader_compound']
✓ Features created
✓ Aggregated to daily sentiment
✓ Saved to data/daily_sentiment.csv

Summary:
  Shape: (40, 4)
  Date range: 2026-02-19 to 2026-03-30

First few rows:
         date  avg_vader_sentiment  avg_finbert_sentiment  tweet_volume
0  2026-02-19             1.810823              -0.021277            47
1  2026-02-20            12.465719              -3.404762            42
2  2026-02-21             1.132041              -0.159091            88
3  2026-02-22            -0.936302              -3.303371            89
4  2026-02-23            -0.033264              -0.191489            47
Feature engineering complete
Daily sentiment aggregation saved to data/daily_sentiment.csv


## Steps 6-7: Price Data, Merge Sentiment & Price, Correlation Analysis

In [10]:
try:
    get_price_data_main()
    print("Price data download complete")
except Exception as e:
    print(f"Warning: {e}")


try:
    merge_sentiment_price_main()
    print("Data merging complete")
except Exception as e:
    print(f"Warning: {e}")


Shape: (40, 7)
Date range: 2026-02-19 00:00:00 to 2026-03-30 00:00:00

First few rows:
Price        Date         Close          High           Low          Open  \
Ticker                  BTC-USD       BTC-USD       BTC-USD       BTC-USD   
0      2026-02-19  66957.523438  67277.125000  65637.429688  66425.625000   
1      2026-02-20  68005.421875  68269.031250  66452.484375  66958.578125   
2      2026-02-21  68003.765625  68657.703125  67533.070312  68000.250000   
3      2026-02-22  67659.390625  68235.226562  67185.601562  67998.828125   
4      2026-02-23  64616.738281  67668.429688  63924.437500  67668.429688   

Price        Volume daily_return  
Ticker      BTC-USD               
0       31492987019          NaN  
1       47507867304     1.565020  
2       18357635642    -0.002435  
3       17893536012    -0.506406  
4       50953457309    -4.497014  
Price data download complete
Shape: (39, 10)

First few rows:
         date  avg_vader_sentiment  avg_finbert_sentiment  tweet_

## Results & Visualizations (Load and analyze final output)

In [11]:
print("\nLoading merged data for visualization...")
merged_df = pd.read_csv('data/merged_final.csv')
merged_df['date'] = pd.to_datetime(merged_df['date'])

df_btc = merged_df.copy()

print(f"Loaded {len(df_btc)} records")
print(f"  Date range: {df_btc['date'].min().date()} to {df_btc['date'].max().date()}")
print(f"  Columns: {df_btc.columns.tolist()}")


Loading merged data for visualization...
Loaded 39 records
  Date range: 2026-02-20 to 2026-03-30
  Columns: ['date', 'avg_vader_sentiment', 'avg_finbert_sentiment', 'tweet_volume', 'Close', 'High', 'Low', 'Open', 'Volume', 'daily_return']


### Visualization 1: Sentiment vs Price Over Time

In [12]:
# Sentiment vs price over time with secondary y-axis
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=df_btc['date'],
    y=df_btc['avg_finbert_sentiment'],
    name='Avg FinBERT',
    yaxis='y1'
))
fig1.add_trace(go.Scatter(
    x=df_btc['date'],
    y=df_btc['daily_return'],
    name='Daily Return (%)',
    yaxis='y2'
))
fig1.add_trace(go.Scatter(
    x=df_btc['date'],
    y=df_btc['avg_vader_sentiment'],
    name='Avg VADER',
    yaxis='y1'
))
fig1.update_layout(
    title='Bitcoin Sentiment vs Daily Return Over Time',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Sentiment Score', side='left'),
    yaxis2=dict(title='Daily Return (%)', overlaying='y', side='right'),
    hovermode='x unified',
    height=500
)
fig1.show()

### Visualization 2 & 3: Correlation Heatmap and Normalized Trends

In [13]:
# Correlation heatmap
corr_data = df_btc[['avg_vader_sentiment', 'avg_finbert_sentiment', 'tweet_volume', 'Close', 'daily_return']].corr()

fig4 = go.Figure(data=go.Heatmap(
    z=corr_data.values,
    x=corr_data.columns,
    y=corr_data.columns,
    text=corr_data.values.round(3),
    texttemplate='%{text}',
    textfont={'size': 10},
    colorscale='RdBu',
    zmid=0,
    zmin=-1,
    zmax=1
))
fig4.update_layout(
    title='Correlation Matrix: Sentiment, Volume, and Price',
    height=500
)
fig4.show()

# Normalized trends
df_plot = df_btc.copy()
df_plot['normalized_finbert_sentiment'] = (df_plot['avg_finbert_sentiment'] - df_plot['avg_finbert_sentiment'].min()) / (df_plot['avg_finbert_sentiment'].max() - df_plot['avg_finbert_sentiment'].min())
df_plot['normalized_vader_sentiment'] = (df_plot['avg_vader_sentiment'] - df_plot['avg_vader_sentiment'].min()) / (df_plot['avg_vader_sentiment'].max() - df_plot['avg_vader_sentiment'].min())
df_plot['normalized_price'] = (df_plot['Close'] - df_plot['Close'].min()) / (df_plot['Close'].max() - df_plot['Close'].min())

fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['normalized_finbert_sentiment'], name='FinBERT Sentiment (normalized)', line=dict(color='blue')))
fig5.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['normalized_vader_sentiment'], name='VADER Sentiment (normalized)', line=dict(color='green')))
fig5.add_trace(go.Scatter(x=df_plot['date'], y=df_plot['normalized_price'], name='Price (normalized)', line=dict(color='orange')))
fig5.update_layout(
    title='Normalized Sentiment vs Price Trend',
    xaxis_title='Date',
    yaxis_title='Normalized Value (0-1)',
    hovermode='x unified',
    height=400
)
fig5.show()